# LFM3-1.7B Uncensored GGUF — llama.cpp chat on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/lfm3-1.7b-uncensored-gguf-48bb/colab/LFM3-1.7B-Uncensored-GGUF-llamacpp.ipynb)

[`AtomicChat/LFM3-1.7B-Uncensored-GGUF`](https://huggingface.co/AtomicChat/LFM3-1.7B-Uncensored-GGUF) is AtomicChat's **safety-ablated / refusal-reduced** GGUF of [`LiquidAI/LFM3-1.7B`](https://huggingface.co/LiquidAI/LFM3-1.7B) (~1.73B, `lfm3` architecture). A projected rank-1 LoRA is baked into the weights, then AD-quantized. The card reports the same MMLU as the base (41.0% on 2000 questions) with refusals on their harmful-prompt set going from 61.7% to 0.0%.

Why this notebook: the repo already has [`colab/LFM3-1.7B-chat-tools.ipynb`](https://github.com/donvito/notebooks/blob/main/colab/LFM3-1.7B-chat-tools.ipynb) for the **base** LiquidAI checkpoint through Transformers + `trust_remote_code`. This one is specifically the **AtomicChat Uncensored GGUF** through **llama.cpp**. The two are not interchangeable — this GGUF is `lfm3` and needs a llama.cpp build that includes [`ggml-org/llama.cpp#31252`](https://github.com/ggml-org/llama.cpp/pull/31252) (tag [`b11938`](https://github.com/ggml-org/llama.cpp/releases/tag/b11938) / commit `19cce69` or newer). Liquid AI's own GGUF card documents [`b11960`](https://github.com/ggml-org/llama.cpp/releases/tag/b11960) / [`v0.8.0`](https://github.com/ggml-org/llama.cpp/releases/tag/v0.8.0) as the supported floor.

This walkthrough is **inference only**. It does not train or fine-tune anything. Demos are ordinary useful chat — not prompts that solicit harmful or illegal content. The uncensored weights still have to be used in line with the base license.

**Licence:** [LFM Open License v1.0](https://huggingface.co/LiquidAI/LFM3-1.7B/blob/main/LICENSE) (`license: other`, `lfm-open-license-v1.0`), same as [`LiquidAI/LFM3-1.7B`](https://huggingface.co/LiquidAI/LFM3-1.7B). This is **not** Apache 2.0 or MIT. Read the license before any production or paid use.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has ~15 GB of VRAM, **no native bfloat16**, and **no FlashAttention-2**. Do not turn FlashAttention on (`-fa off`). Default weights are **AD-Q4_K_M** (~1.1 GB); **AD-Q6_K** (~1.6 GB) is an optional quality bump that still fits. Context is capped at **8,192** tokens — the card's 256K window is not a T4 demo. Weights plus that cache land around 2–4 GB on 15 GB.

**Links:** [AtomicChat GGUF](https://huggingface.co/AtomicChat/LFM3-1.7B-Uncensored-GGUF) · [base card](https://huggingface.co/LiquidAI/LFM3-1.7B) · [official GGUF](https://huggingface.co/LiquidAI/LFM3-1.7B-GGUF) · [LFM3 blog](https://www.liquid.ai/blog/lfm3) · [llama.cpp #31252](https://github.com/ggml-org/llama.cpp/pull/31252) · [Transformers sibling](https://github.com/donvito/notebooks/blob/main/colab/LFM3-1.7B-chat-tools.ipynb)


## 1. Install an LFM3-capable llama.cpp (official CUDA binary)

Published `llama-cpp-python` CUDA wheels are still **0.3.36** (1 October 2026). That release vendors llama.cpp `@0c1e57098`, which **predates** LFM3 support (merged 27 October 2026 in #31252 / `b11938`). The 0.3.34 / 0.3.36 `cu121` wheel will fail to load this GGUF (`unknown model architecture: lfm3`). There is no newer `llama-cpp-python` release as of this notebook.

Building `llama-cpp-python` from git against a bumped `vendor/llama.cpp` would be the nicest Python API, but it is a poor Colab fit: 15–25 minutes of CUDA compile on a T4, plus ctypes drift between the 0.3.36 bindings and llama.cpp `b11960` / `v0.8.0`.

**Path chosen:** download the official ggml-org **CUDA 12.8** Linux tarball for **`b11960`** (same commit family as the `v0.8.0` stable release; Liquid AI's documented LFM3 floor) and drive chat through `llama-server` + its OpenAI-compatible HTTP API. That is a known-good LFM3 binary, downloads in about a minute (~183 MB), and needs no session restart.

CUDA 12.1/12.8 binaries dynamically link `libcudart.so.12` and `libcublas.so.12`. Current free Colab images often ship the NVIDIA driver but not those user-space libraries, so this cell also installs `nvidia-cuda-runtime-cu12` and `nvidia-cublas-cu12` (same pattern as [`Smaug-Humanizer-llamacpp.ipynb`](https://github.com/donvito/notebooks/blob/main/colab/Smaug-Humanizer-llamacpp.ipynb)). Use `%pip` (not `python -I`) so Colab's site-packages stay on the path.

The card's one-liner is the same binary:

```bash
llama-cli -hf AtomicChat/LFM3-1.7B-Uncensored-GGUF:AD-Q4_K_M -p "Hello"
```


In [ ]:
%pip install -qqq --no-cache-dir nvidia-cuda-runtime-cu12 nvidia-cublas-cu12 huggingface_hub

import os
import tarfile
import urllib.request

# Official ggml-org build with LFM3 (#31252). b11960 == v0.8.0 binary tag.
# Do NOT use the llama-cpp-python 0.3.34 / 0.3.36 cu121 wheel for this GGUF.
LLAMA_TAG = "b11960"
LLAMA_URL = (
    "https://github.com/ggml-org/llama.cpp/releases/download/"
    f"{LLAMA_TAG}/llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"
)
ROOT = "/content" if os.path.isdir("/content") else "/tmp"
BIN = os.path.join(ROOT, f"llama-{LLAMA_TAG}")
ARCHIVE = os.path.join(ROOT, f"llama-{LLAMA_TAG}-cuda.tgz")

cli = os.path.join(BIN, "llama-cli")
server = os.path.join(BIN, "llama-server")
if not (os.path.exists(cli) and os.path.exists(server)):
    print("downloading", LLAMA_URL)
    urllib.request.urlretrieve(LLAMA_URL, ARCHIVE)
    with tarfile.open(ARCHIVE) as tar:
        tar.extractall(ROOT)  # creates ROOT/llama-b11960/{llama-cli,llama-server,lib*.so}

assert os.path.exists(cli), f"llama-cli missing under {BIN}"
assert os.path.exists(server), f"llama-server missing under {BIN}"
print("llama.cpp", LLAMA_TAG, "->", BIN)
print("llama-cli:", cli)
print("llama-server:", server)



## 2. Check the GPU and print T4 caveats

Confirm a T4 (or better) is attached, preload CUDA 12 user-space libs so the official binary can find `libcudart.so.12` / `libcublas.so.12`, and print `nvidia-smi` used/free/total. llama.cpp allocates CUDA memory outside PyTorch, so `nvidia-smi` is the measured counter.


In [ ]:
import ctypes
import glob
import importlib
import os
import subprocess

import torch


def _preload_cuda12_runtime() -> list[str]:
    """Make CUDA 12 runtime/cublas visible before starting llama-server.

    The official CUDA 12.8 tarball CDLLs libggml-cuda, which needs
    libcudart.so.12 and libcublas.so.12. Those live in the nvidia-*-cu12
    pip packages on current free Colab, and sometimes under /usr/local/cuda.
    """
    sos = []
    for pkg_name in ("nvidia.cuda_runtime", "nvidia.nvjitlink", "nvidia.cublas"):
        try:
            pkg = importlib.import_module(pkg_name)
        except ImportError:
            continue
        sos.extend(glob.glob(os.path.join(pkg.__path__[0], "lib", "*.so*")))
    for so in sos + sos:
        try:
            ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
        except OSError:
            continue
    return sos


def cuda_lib_dirs() -> list[str]:
    found = []
    for pattern in (
        "/usr/local/cuda*/lib64",
        "/usr/local/cuda*/targets/x86_64-linux/lib",
        "/usr/lib/x86_64-linux-gnu",
        "/usr/local/lib/python*/dist-packages/nvidia/cuda_runtime/lib",
        "/usr/local/lib/python*/dist-packages/nvidia/cublas/lib",
        "/usr/local/lib/python*/dist-packages/nvidia/nvjitlink/lib",
    ):
        found.extend(glob.glob(pattern))
    return [path for path in found if os.path.isdir(path)]


def print_vram(label=""):
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.free,memory.total", "--format=csv,noheader"],
        text=True,
    ).strip()
    suffix = f" ({label})" if label else ""
    print(f"nvidia-smi used/free/total{suffix}: {smi}")


preloaded = _preload_cuda12_runtime()
LIBS = cuda_lib_dirs()
for pkg_name in ("nvidia.cuda_runtime", "nvidia.nvjitlink", "nvidia.cublas"):
    try:
        pkg = importlib.import_module(pkg_name)
    except ImportError:
        continue
    lib_dir = os.path.join(pkg.__path__[0], "lib")
    if os.path.isdir(lib_dir) and lib_dir not in LIBS:
        LIBS.append(lib_dir)
ENV = os.environ.copy()
ENV["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
ENV["LD_LIBRARY_PATH"] = ":".join([BIN, *LIBS, ENV.get("LD_LIBRARY_PATH", "")])

if not any(os.path.exists(os.path.join(path, "libcudart.so.12")) for path in [BIN, *LIBS]):
    raise SystemExit(
        "This CUDA 12.8 llama.cpp build needs libcudart.so.12. "
        "Re-run the install cell (nvidia-cuda-runtime-cu12) or check /usr/local/cuda/lib64."
    )

print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

!nvidia-smi

major, minor = torch.cuda.get_device_capability()
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
name = torch.cuda.get_device_name()
# Turing (T4, sm_75) has no FlashAttention-2.
FA = "on" if major >= 8 else "off"

print()
print("=== T4-first caveats ===")
print(f"GPU: {name}  sm_{major}{minor}  VRAM: {vram_gb:.1f} GB")
print(f"llama.cpp {LLAMA_TAG} (official CUDA 12.8 binary, LFM3 via #31252)")
print(f"-fa {FA}  |  preloaded CUDA 12 .so files: {len(preloaded)}")
print("- No native bfloat16 on T4 (sm_75). This notebook loads AD-Q4_K_M GGUF, not BF16.")
print("- No FlashAttention-2 on sm_75. flash_attn=False / -fa off.")
print("- AD-Q4_K_M is ~1.1 GB; 8K context still fits on 15 GB with a lot of headroom.")
print("- AD-Q6_K (~1.6 GB) is optional if you want a larger quant on the same T4.")
print("- Do not install llama-cpp-python 0.3.34 / 0.3.36 — those wheels cannot load lfm3.")
print("- Distinct from colab/LFM3-1.7B-chat-tools.ipynb (base LiquidAI weights via Transformers).")
print_vram("before load")
if major < 8:
    print("- FlashAttention stays off on this GPU.")



## 3. Load AD-Q4_K_M and start `llama-server`

`hf_hub_download` pulls `LFM3-1.7B-Uncensored-AD-Q4_K_M.gguf` (~1.1 GB) from [`AtomicChat/LFM3-1.7B-Uncensored-GGUF`](https://huggingface.co/AtomicChat/LFM3-1.7B-Uncensored-GGUF). `n_gpu_layers=-1` maps to `llama-server -ngl all` (b11960's spelling for every layer). `n_ctx=8192` is plenty for 1.1 GB on 15 GB. `flash_attn=False` (`-fa off`) is required on sm_75.

To try **AD-Q6_K** (~1.6 GB) instead, change `FILENAME` to `LFM3-1.7B-Uncensored-AD-Q6_K.gguf`.

Sampler defaults match the [LFM3 card](https://huggingface.co/LiquidAI/LFM3-1.7B) chat / tools / agents recipe: temperature 0.1, top_k 50, repeat penalty 1.1. Reasoning is **on by default** in LFM3; this notebook turns it off (`enable_thinking=false`) so free-T4 demos stay short.

`HF_HUB_DISABLE_IMPLICIT_TOKEN=1` — the repo is public. Cancel any Colab `HF_TOKEN` prompt.


In [ ]:
import os
import subprocess
import time

import requests
from huggingface_hub import hf_hub_download

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

REPO_ID = "AtomicChat/LFM3-1.7B-Uncensored-GGUF"
FILENAME = "LFM3-1.7B-Uncensored-AD-Q4_K_M.gguf"
# Optional larger quant on the same T4 (~1.6 GB):
# FILENAME = "LFM3-1.7B-Uncensored-AD-Q6_K.gguf"

N_CTX = 8192
N_GPU_LAYERS = -1  # llama-cpp-python style; b11960 CLI wants -ngl all
FLASH_ATTN = False  # T4 / sm_75 — maps to -fa off
NGL = "all" if N_GPU_LAYERS < 0 else str(N_GPU_LAYERS)
HOST = "127.0.0.1"
PORT = 8088
SERVER_URL = f"http://{HOST}:{PORT}"

SAMPLER = dict(temperature=0.1, top_k=50, repeat_penalty=1.1)
CHAT_TEMPLATE_KWARGS = {"enable_thinking": False}

MODEL = hf_hub_download(REPO_ID, FILENAME)
print("model", MODEL)

# Re-run safe: stop a leftover server from an earlier cell.
if "server_proc" in globals() and server_proc is not None and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        server_proc.kill()

LOG = os.path.join(ROOT, "llama-server.log")
log_f = open(LOG, "w")
cmd = [
    os.path.join(BIN, "llama-server"),
    "-m", MODEL,
    "-ngl", NGL,
    "-c", str(N_CTX),
    "-fa", FA if FLASH_ATTN else "off",
    "--host", HOST,
    "--port", str(PORT),
    "--jinja",
    "--reasoning", "off",
    "--chat-template-kwargs", '{"enable_thinking":false}',
    "--temp", str(SAMPLER["temperature"]),
    "--top-k", str(SAMPLER["top_k"]),
    "--repeat-penalty", str(SAMPLER["repeat_penalty"]),
]
print("starting", " ".join(cmd))
server_proc = subprocess.Popen(
    cmd,
    env=ENV,
    stdout=log_f,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
)

deadline = time.time() + 180
last_err = None
while time.time() < deadline:
    if server_proc.poll() is not None:
        log_f.flush()
        raise SystemExit(
            f"llama-server exited {server_proc.returncode}. Tail of {LOG}:\n"
            + open(LOG).read()[-2000:]
        )
    try:
        health = requests.get(f"{SERVER_URL}/health", timeout=2)
        if health.ok:
            print("llama-server health:", health.text.strip())
            break
        last_err = f"HTTP {health.status_code} {health.text[:200]}"
    except requests.RequestException as exc:
        last_err = str(exc)
    time.sleep(1)
else:
    raise SystemExit(f"llama-server did not become healthy: {last_err}\n" + open(LOG).read()[-2000:])

print()
print(
    "loaded", FILENAME,
    "| n_ctx", N_CTX,
    "| n_gpu_layers", N_GPU_LAYERS,
    "| flash_attn", FLASH_ATTN,
)
print("sampler", SAMPLER, "| chat_template_kwargs", CHAT_TEMPLATE_KWARGS)
print("server", SERVER_URL)
print()
print_vram("after load")
print()
print("GPU status after load:")
!nvidia-smi



## 4. Chat helper

One user turn through `llama-server`'s `/v1/chat/completions`. The GGUF already carries the LFM3 ChatML template (`--jinja`). Thinking stays off. Demos cap completions at **256** tokens so a free T4 finishes them in one session.


In [ ]:
def chat(prompt: str, max_tokens: int = 256, system: str | None = None) -> str:
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    payload = {
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": SAMPLER["temperature"],
        "top_k": SAMPLER["top_k"],
        "repeat_penalty": SAMPLER["repeat_penalty"],
        "chat_template_kwargs": CHAT_TEMPLATE_KWARGS,
    }
    response = requests.post(
        f"{SERVER_URL}/v1/chat/completions",
        json=payload,
        timeout=180,
    )
    response.raise_for_status()
    data = response.json()
    choice = data["choices"][0]
    text = (choice.get("message") or {}).get("content") or ""
    usage = data.get("usage") or {}
    print(
        f"prompt {usage.get('prompt_tokens', '?')} | "
        f"completion {usage.get('completion_tokens', '?')} | "
        f"finish {choice.get('finish_reason')}"
    )
    print_vram("after chat")
    return text.strip()


print("chat helper ready | max_tokens default 256 | thinking off")



## 5. Demo 1 — short witty fiction

A 120-word scene, not a long story. Ordinary creative chat; nothing the base model would need to refuse.


In [ ]:
DEMO_1 = (
    "Write a 120-word scene: a raccoon in a tiny raincoat negotiates a midnight "
    "truce with a very tired barista. Witty, not cute-for-its-own-sake. End on "
    "one good line of dialogue."
)
print("=== prompt ===")
print(DEMO_1)
print()
print("=== reply ===")
print(chat(DEMO_1))



## 6. Demo 2 — technical Q&A

Explain a coding concept the way you would to a junior engineer. Card sampler stays on the chat / tools recipe (low temperature).


In [ ]:
DEMO_2 = (
    "Explain Python generators versus lists to a junior engineer. Cover memory, "
    "one-pass iteration, and when a list is still the right choice. Use one "
    "short code example and keep the whole answer under 200 words."
)
print("=== prompt ===")
print(DEMO_2)
print()
print("=== reply ===")
print(chat(DEMO_2))



## 7. Demo 3 — fictional cozy-mystery outline

A mild creative-writing prompt a heavily safety-tuned chat model sometimes hedges on, still fully legal and fictional: plot beats for a novella, not real-world crime advice.


In [ ]:
DEMO_3 = (
    "Outline a 5-beat plot for a cozy mystery novella about a stolen sourdough "
    "starter at a village bakery. Keep it witty. No real-world crime advice — "
    "this is fiction for a story workshop."
)
print("=== prompt ===")
print(DEMO_3)
print()
print("=== reply ===")
print(chat(DEMO_3))



## 8. Caveats and cleanup

**What this notebook is for.** Inference-only chat on a free T4 with AtomicChat's AD-Q4_K_M Uncensored GGUF and an official llama.cpp CUDA build that already has LFM3 (`b11960` / `v0.8.0`). The three demos are short on purpose.

**What it is not.** The [Transformers LFM3-1.7B notebook](https://github.com/donvito/notebooks/blob/main/colab/LFM3-1.7B-chat-tools.ipynb) (base LiquidAI weights, thinking, tool loop). Training. A how-to for harm. The uncensored GGUF removes many safety refusals; use it responsibly and in line with the [LFM Open License v1.0](https://huggingface.co/LiquidAI/LFM3-1.7B/blob/main/LICENSE).

**Why not `llama-cpp-python` 0.3.36.** That CUDA wheel vendors llama.cpp from 1 October 2026 and cannot load `architecture = lfm3`. A from-source Python build against #31252 is possible but is a long, brittle Colab compile. Official `llama-server` is the Colab-workable path.

**T4 notes.** AD-Q4_K_M (~1.1 GB) plus `n_ctx=8192` is the free-T4 budget, with plenty of headroom. AD-Q6_K (~1.6 GB) is the next rung on the same 15 GB card. Do not load BF16. `-fa off` on sm_75. `print_vram` reports `nvidia-smi` used/free/total because llama.cpp does not go through `torch.cuda.memory_allocated`.

**Thinking.** LFM3 reasoning is on by default. This notebook passes `enable_thinking=false` so demos stay short. Turn it back on for harder questions if you want the `<think>` trace.

**Licence.** LFM Open License v1.0, same as [`LiquidAI/LFM3-1.7B`](https://huggingface.co/LiquidAI/LFM3-1.7B). Not Apache / MIT.

**Links again:** [AtomicChat GGUF](https://huggingface.co/AtomicChat/LFM3-1.7B-Uncensored-GGUF) · [base card](https://huggingface.co/LiquidAI/LFM3-1.7B) · [official GGUF](https://huggingface.co/LiquidAI/LFM3-1.7B-GGUF) · [LFM3 blog](https://www.liquid.ai/blog/lfm3) · [llama.cpp #31252](https://github.com/ggml-org/llama.cpp/pull/31252)

**Not executed here.** The agent VM has no GPU. Install cells parse; a full T4 `llama-server` load was not run in this environment.

**Do not merge from this agent.** Leave the PR open for human review.

Free the GPU before you open another notebook:


In [ ]:
print_vram("before cleanup")
if "server_proc" in globals() and server_proc is not None and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        server_proc.kill()
        server_proc.wait(timeout=5)
    print("llama-server stopped")
else:
    print("llama-server was not running")
print_vram("after cleanup")

